# 🧬 02 — Pydantic from zero: shapes that refuse to be wrong

Every byte that crosses a package boundary in this repo — Ada's request, Bell's offer,
the controller's answers — travels as a **pydantic model**: a Python class that
*validates its own fields the moment it is built*. This notebook teaches pydantic from
zero, entirely on the repo's real shapes in `interfaces/src/a2a_interfaces/models.py`.

**What you'll be able to do after this notebook**

- declare a validated shape with `BaseModel` and read a `ValidationError` like a native
- explain why every cross-package payload here is frozen, typo-rejecting, and range-checked to the chain's integer widths
- parse a loose dict into the right class with a discriminated union (`TypeAdapter` + the `kind` tag)
- round-trip models through JSON — snake_case vs camelCase, and bytes-as-hex
- name the one hole in the armor (`model_copy` skips validation) and the discipline that keeps it harmless

**You need:** notebooks `00_orientation` (repo map, imports) and `01_python_toolbox`
(classes, `Annotated`, `Literal`, `try/except`). No infrastructure — the `interfaces`
package is pure data, so this whole chapter runs offline.

**Estimated time:** 60–75 minutes.

> **How to run:** ▶▶ Run All, or step with **Shift+Enter**. Every code cell is preceded
> by a markdown cell telling you what to look for in the output.

## 0. Setup — import the cast

Three imports matter (you met importing in `00_orientation`): `pydantic` itself, the
repo's real `models` module, and `fixtures as fx` — the one canonical
Ada/Bell/ticket-#7 example that story, docs, and tests all share. We also locate the
repo root so we can point at real source files by path.

In [ ]:
import inspect
import json
from pathlib import Path
from typing import Annotated

from pydantic import BaseModel, Field, StringConstraints, TypeAdapter, ValidationError

from a2a_interfaces import fixtures as fx
from a2a_interfaces import models

ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())
MODELS_FILE = Path(models.__file__)
print("shapes live in →", MODELS_FILE.relative_to(ROOT))
print("the cast       → Ada", fx.ADA[:10] + "…", "| Bell", fx.BELL[:10] + "…", "| ticket #" + str(fx.TICKET_ID))

## 1. The horror story: two packages, one bare dict

Before pydantic, watch the disease it cures. Package A builds a plain dict; package B
reads it. They agree on the field names *only in their heads*. Below, the provider has a
one-letter typo — and Ada is granted **0 bps** instead of 50 Mbps. Look for what's
missing in the output: **any error at all**.

In [ ]:
def provider_quote():                       # imagine this lives in Bell's package…
    return {"capactiy_bps": 50_000_000}     # …with a typo nobody will ever see


def consumer_grant(quote):                  # …and this lives in Ada's package
    return quote.get("capacity_bps", 0)     # reading the *correct* spelling


granted = consumer_grant(provider_quote())
print("provider meant → 50,000,000 bps")
print("consumer got   →", granted, "bps      ← no crash, no warning: a silent lie")
assert granted == 0                          # the bug "works"

**✏️ Your turn 1.1 — fix the typo, watch the lie die**

Rewrite the provider's quote below with the key spelled the way the consumer reads it,
re-run, and watch the granted number change. Success: you see exactly 50,000,000 bps
and the un-commented assert passes.

In [ ]:
def my_provider_quote():
    return {"capactiy_bps": 50_000_000}    # TODO: fix the key the consumer actually reads


my_granted = consumer_grant(my_provider_quote())
print("consumer got →", my_granted, "bps")
# assert my_granted == 50_000_000   # un-comment once fixed

<details><summary>✅ Solution 1.1 — peek only after trying</summary>

```python
def my_provider_quote():
    return {"capacity_bps": 50_000_000}

my_granted = consumer_grant(my_provider_quote())
assert my_granted == 50_000_000
```

The fix is one letter — which is the point: nothing but human spelling care held the two
packages together. §2 hands that job to a class.
</details>

## 2. `BaseModel`: a class whose fields defend themselves

The fix: declare the shape **once, as code**, and validate every payload against it *at
the boundary* — the moment data enters your package. A pydantic model is a class (01)
whose fields are type hints (01); its constructor checks every field. Pydantic is also
friendly about *coercion*: a `"7"` that clearly means the integer `7` gets converted,
not rejected. Watch the `repr()`s — strings go in, real types come out.

In [ ]:
class Ticket(BaseModel):        # a toy — the repo's real shapes arrive in §4
    id: int
    owner: str
    active: bool


t = Ticket(id="7", owner="Ada", active="true")     # strings in…
print("t.id     →", repr(t.id), "     ← coerced str → int")
print("t.active →", repr(t.active), "  ← coerced str → bool")
assert t.id == 7 and t.active is True
t                                # last expression: the notebook displays it

**✏️ Your turn 2.1 — where coercion draws the line**

Coercion converted `"7"` to `7` — how far does the friendliness go? Write your two
predictions into the TODO comment *before* running: is `id=7.0` accepted, and is
`id=7.5`? Success: the printed lines match your predictions and the assert passes.

In [ ]:
# TODO: predict BEFORE running (accepted / rejected):
#   id=7.0 → ___        id=7.5 → ___
for candidate in (7.0, 7.5):
    try:
        tk = Ticket(id=candidate, owner="Ada", active=True)
        print(f"id={candidate!r} → accepted, tk.id == {tk.id!r}")
    except ValidationError as e:
        print(f"id={candidate!r} → ✗ {e.errors()[0]['type']}")
# assert Ticket(id=7.0, owner="Ada", active=True).id == 7   # un-comment after predicting

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
# id=7.0 → accepted (tk.id == 7)     id=7.5 → ✗ int_from_float
assert Ticket(id=7.0, owner="Ada", active=True).id == 7
```

Coercion converts only when no information is lost: `7.0` *is* the integer 7, but
turning `7.5` into an int would silently drop the half — so pydantic refuses.
</details>

Feed it something that *can't* be an `int` and construction fails with a
`ValidationError` — an exception (01) that carries structured details, not just a
message. We catch it with `try/except` so the notebook keeps running; look at the four
fields it reports for the failure.

In [ ]:
try:
    Ticket(id="seven", owner="Ada", active=True)
except ValidationError as e:
    print("✗ rejected with", e.error_count(), "error\n")
    err = e.errors()[0]
    for key in ("loc", "type", "msg", "input"):
        print(f"  {key:<5} → {err[key]!r}")

## 3. Reading a `ValidationError` out loud

Every entry in `e.errors()` has the same anatomy:

- **`loc`** — the *path* to the bad field, as a tuple (nested fields chain up: `('window', 'start')`)
- **`type`** — a machine-readable code (`int_parsing`, `missing`, `extra_forbidden`, …)
- **`msg`** — the human-readable sentence
- **`input`** — the exact value that offended

Read the one above out loud: *"at `id`: the input `'seven'` failed `int_parsing` — not
a valid integer."* You will read dozens of these; the `type` code is the part to trust.
One more habit: pydantic reports **all** problems in one construction, not just the
first — three bad fields, three entries.

In [ ]:
try:
    Ticket(id="seven", owner=42, active="maybe")   # three bad fields at once
except ValidationError as e:
    print("one construction, every problem reported:")
    for err in e.errors():
        loc = ".".join(str(part) for part in err["loc"])
        print(f"  {loc:<8} → {err['type']}")
    assert e.error_count() == 3

**✏️ Your turn 3.1 — two absences, one type code**

Build a `Ticket` supplying *only* `id` and read the report out loud, §3-style. Then
answer in code: which `type` code do both errors share? Success: the un-commented
assert passes with your answer.

In [ ]:
caught = None
try:
    Ticket(id="7")                       # owner and active never arrive
except ValidationError as e:
    caught = e
    for err in e.errors():
        print(f"  {err['loc'][0]:<7} → {err['type']}")

shared_type = "..."   # TODO: the one `type` code both errors share
# assert {err["type"] for err in caught.errors()} == {shared_type}   # un-comment once answered

<details><summary>✅ Solution 3.1 — peek only after trying</summary>

```python
shared_type = "missing"
assert {err["type"] for err in caught.errors()} == {shared_type}
```

A field that never arrived is its own error code — `missing` — distinct from a field
that arrived broken (`int_parsing`). The `loc` tells you *which* absence.
</details>

## 4. Integers with edges: `Field(ge=…, le=…)` → `Uint8`/`Uint64`

A bare `int` happily accepts `-5` and `10**100`. Real fields have *ranges*. Pydantic
attaches range rules with `Field(ge=…, le=…)` (**g**reater-or-**e**qual /
**l**ess-or-**e**qual) inside `Annotated` — the "type hint with extra baggage" from 01.
Toy first: a dice roll. `TypeAdapter` is new here — it wraps a *bare type* (no class
needed) and gives it `.validate_python()`; remember it, §9's union needs it too.

In [ ]:
Dice = Annotated[int, Field(ge=1, le=6)]     # an int, plus rules riding along
roll = TypeAdapter(Dice)                     # a validator for a bare type

print("roll 6 →", roll.validate_python(6), " ✓")
try:
    roll.validate_python(7)
except ValidationError as e:
    print("roll 7 → ✗", e.errors()[0]["type"])

Now the real thing. The repo names three of these, one per **chain integer width**:
Solidity (the smart-contract language, notebook 06) has fixed-size unsigned integers —
a `uint8` slot holds 0…255, a `uint64` slot holds 0…2⁶⁴−1. Python ints are unbounded,
so these aliases teach Python the chain's limits. This is real source from `models.py` —
note they are *type aliases* (a name assigned to a type), not classes:

In [ ]:
src = MODELS_FILE.read_text()
print(src[src.index("# unsigned integers") : src.index("# 0x-prefixed hex")].rstrip())
print()
print("uint64 max →", f"{2**64 - 1:,}", f"({(2**64 - 1).bit_length()} bits)")

Poke the exact edge, then break it on the real model: a `capacity_bps` of `2**64` would
not fit the chain's slot — and it dies **here, in Python**, long before any transaction
could carry it toward Solidity. The border is the cheap place to catch overflow.

In [ ]:
u64 = TypeAdapter(models.Uint64)
edge = 2**64 - 1
print("exactly 2**64 - 1 → accepted ✓" if u64.validate_python(edge) == edge else "?")

try:
    models.BandwidthNeed(src="hostA", dst="hostB",
                         capacity_bps=2**64,          # one past the chain's width
                         qos_class=1, window=fx.WINDOW)
except ValidationError as e:
    err = e.errors()[0]
    print(f"capacity_bps=2**64 → ✗ {err['type']}   ← caught before Solidity ever sees it")
    assert err["type"] == "less_than_equal"

**✏️ Your turn 4.1 — build the qos_class edge-checker**

`BandwidthNeed.qos_class` is a `Uint8` — one chain byte, 0…255. Build the adapter and
probe its far edge the way the cell above probed `Uint64`: 255 in, then 256 inside a
`try/except` (§4's roll-a-7 cell is your template). Success: 255 passes and 256 names
`less_than_equal`.

In [ ]:
u8 = TypeAdapter(models.Uint8)
print("qos_class 255 →", u8.validate_python(255), " ✓ the widest uint8")

# TODO: validate 256 inside try/except ValidationError and print the error `type`

# assert u8.validate_python(2**8 - 1) == 255   # un-comment: the edge holds

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
u8 = TypeAdapter(models.Uint8)
try:
    u8.validate_python(256)
except ValidationError as e:
    print("256 →", e.errors()[0]["type"])      # less_than_equal
assert u8.validate_python(2**8 - 1) == 255
```

Same lesson as `capacity_bps`, one byte smaller: the alias teaches Python the chain's
slot width, so overflow dies at the border instead of inside a transaction.
</details>

## 5. Strings with a shape: `StringConstraints` → `Address`, `Signature`, `DecimalString`

Numbers get ranges; strings get **patterns**. `StringConstraints(pattern=…)` takes a
*regular expression* ("regex") — a mini-language for describing string shapes: `^` start
of string, `$` end, `[0-9a-fA-F]` any one hex character, `{4}` exactly four repeats.
Toy: a string that must be `0x` plus exactly four hex chars.

In [ ]:
Hex4 = Annotated[str, StringConstraints(pattern=r"^0x[0-9a-fA-F]{4}$")]
h4 = TypeAdapter(Hex4)

print("'0xBEEF' →", h4.validate_python("0xBEEF"), " ✓")
try:
    h4.validate_python("0xBEE")                     # one char short
except ValidationError as e:
    print("'0xBEE'  → ✗", e.errors()[0]["type"])

The repo's real string aliases, straight from `models.py`. Quick glosses — the deep
versions come later: an **address** is an account's id on the chain, 20 raw bytes
written as hex (notebook 06 earns this properly); a **signature** is 65 bytes proving a
key-holder approved some exact content (notebook 07 makes real ones). A *byte* prints as
exactly **two** hex characters — that's why every count below is doubled.

In [ ]:
print(src[src.index("# 0x-prefixed hex") : src.index("class _Frozen")].rstrip())

Count the characters live on the canonical values — 20 bytes → 40 hex chars for Ada's
address, 65 bytes → 130 for the (placeholder) signature — then break the rule: a
truncated address must never survive a package boundary.

In [ ]:
print("Ada's address →", fx.ADA)
print("hex chars     →", len(fx.ADA) - 2, "  = 20 bytes × 2 chars/byte")
assert len(fx.ADA) - 2 == 40

sig = fx.CANONICAL_SIGNED_OFFER.signature
print("signature     →", sig[:18] + "…  ", len(sig) - 2, "hex chars = 65 bytes (07 explains why 65)")
assert len(sig) - 2 == 130

try:
    models.Offer(**{**fx.CANONICAL_OFFER.model_dump(), "provider": "0x123"})
except ValidationError as e:
    print("provider='0x123' → ✗", e.errors()[0]["type"])

**✏️ Your turn 5.1 — mangle Bell's address**

Wrap `models.Address` in a `TypeAdapter` (§4's trick works on any alias) and feed it
Bell's address with the last character chopped off. Success: the intact address passes
and the mangled one names `string_pattern_mismatch`.

In [ ]:
addr = TypeAdapter(models.Address)
print("Bell intact →", addr.validate_python(fx.BELL)[:12] + "…", "✓")

mangled = fx.BELL          # TODO: chop the last character — fx.BELL[:-1]
try:
    addr.validate_python(mangled)
    print("mangled     → accepted (mangle it first!)")
except ValidationError as e:
    print("mangled     → ✗", e.errors()[0]["type"])
    # assert e.errors()[0]["type"] == "string_pattern_mismatch"   # un-comment once it fires

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
mangled = fx.BELL[:-1]     # 39 hex chars — one short of a real address
```

The regex demands *exactly* 40 hex characters — an address is 20 bytes or it is nothing,
and a truncated one dies at the border instead of pointing at the wrong account.
</details>

`DecimalString` is the strangest one: **money as a string of digits**. Why not a float?
Because floats are *approximations* — they can't even represent 0.1 exactly, and money
must never be approximate. So token amounts travel as decimal strings of the chain's
integer base units (wei-style, 18 decimals — notebook 04 does the full units story), and
the pattern `^[0-9]+$` bans the decimal point outright.

In [ ]:
print("0.1 + 0.2 →", 0.1 + 0.2, "     ← floats can't even do pocket change")

print("price on the wire →", repr(fx.PRICE_10_TOK), "=", int(fx.PRICE_10_TOK) // 10**18, "TOK")

try:
    models.Offer(**{**fx.CANONICAL_OFFER.model_dump(), "price": "10.5"})
except ValidationError as e:
    print("price='10.5'      → ✗", e.errors()[0]["type"], "  ← digits only — no dot, ever")

## 6. `_Frozen`: immutable and typo-proof, by inheritance

Almost every shape in `models.py` subclasses one tiny base. `model_config =
ConfigDict(…)` is pydantic's settings knob — a class attribute that tunes how the model
behaves. Two knobs here: `frozen=True` (instances can never be changed after
construction) and `extra="forbid"` (unknown field names are an *error*, not a shrug).
A name starting with `_` is internal by convention — we peek only to learn.

In [ ]:
print(inspect.getsource(models._Frozen))
print("every shape below it inherits both knobs — the config is written once (inheritance, 01)")

**Frozen** first: try to mutate the canonical window. Note the error is a pydantic
`ValidationError` of type `frozen_instance` — *not* an `AttributeError` — because
pydantic treats mutation as a validation event. Why the repo insists on this: these
instances are shared facts (fixtures, chain views); if one test could quietly edit
`fx.WINDOW`, every later test would inherit the lie.

In [ ]:
try:
    fx.WINDOW.start = 0
except ValidationError as e:
    print("✗ mutation refused →", e.errors()[0]["type"])
    assert e.errors()[0]["type"] == "frozen_instance"
print("window intact →", fx.WINDOW.start, "✓")

**✏️ Your turn 6.1 — try to zero out Ada's bandwidth**

`fx.BANDWIDTH_NEED` inherits the same armor as the window. Predict the exact error
`type` *before* running, then check. Success: your prediction matches and the fixture
still reads 50,000,000 bps.

In [ ]:
# TODO: predict BEFORE running — which error `type` fires? prediction: ___
try:
    fx.BANDWIDTH_NEED.capacity_bps = 0      # try to zero out Ada's bandwidth
except ValidationError as e:
    print("✗ refused →", e.errors()[0]["type"])
    # assert e.errors()[0]["type"] == "frozen_instance"   # un-comment after predicting
print("Ada still gets →", fx.BANDWIDTH_NEED.capacity_bps, "bps ✓")

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
# prediction: frozen_instance — mutation is a validation event, not an AttributeError
assert e.errors()[0]["type"] == "frozen_instance"
```

Same `frozen_instance` as the window — one `_Frozen` base, every shape armored. That is
what keeps a shared fixture a *fact* rather than whatever the last test left behind.
</details>

**`extra="forbid"`** is the horror-story killer. Replay §1's typo against the real
`BandwidthNeed`: instead of a silent 0, the border reports **two** loud errors — the
correct field is `missing` *and* the typo'd one is `extra_forbidden`. Drift cannot hide
behind a validated boundary.

In [ ]:
quote = fx.BANDWIDTH_NEED.model_dump()
quote["capactiy_bps"] = quote.pop("capacity_bps")      # the very typo from §1

try:
    models.BandwidthNeed(**quote)
except ValidationError as e:
    print("§1's silent bug, now impossible to miss:")
    for err in e.errors():
        print(f"  {err['loc'][0]:<14} → {err['type']}")
    assert {err["type"] for err in e.errors()} == {"missing", "extra_forbidden"}

And the inheritance is provable: `BandwidthNeed`'s class lineage (its *method resolution
order*) runs through `_Frozen`, and its config **is** the shared one — one base class,
one place to change the policy for every shape at once.

In [ ]:
print("lineage →", " → ".join(b.__name__ for b in models.BandwidthNeed.__mro__[:3]))
print("frozen  →", models.BandwidthNeed.model_config["frozen"],
      "| extra →", models.BandwidthNeed.model_config["extra"])
assert models.BandwidthNeed.model_config == models.TimeWindow.model_config

## 7. Tour: the shapes Ada and Bell actually exchange

Now read the real cast. `TimeWindow` is the validity window — absolute **unix seconds**
(seconds counted since 1970-01-01 UTC; notebook 04 makes these numbers human). Two facts
to notice below: models with equal fields compare equal with `==`, and the canonical
window is just a `TimeWindow` anyone can rebuild.

In [ ]:
w = models.TimeWindow(start=1757944800, end=1757952000)
print("hand-built == fx.WINDOW →", w == fx.WINDOW, "  ← pydantic compares field-by-field")
assert w == fx.WINDOW
print()
print(inspect.getsource(models.TimeWindow))

`BandwidthNeed` / `TelemetryNeed` — what a consumer *asks for*. Read the sources: the
`v` and `kind` fields are `Literal`s with defaults (both explained in §9), and the
inline comments carry a boundary rule — `src`/`dst`/`target` are names from the
provider's public *catalog*, never real device names. The name→topology mapping stays
the controller's secret (ADR-005; notebook 08).

In [ ]:
print(inspect.getsource(models.BandwidthNeed))
print(inspect.getsource(models.TelemetryNeed))

**✏️ Your turn 7.1 — build a TelemetryNeed for leafB**

You just read `TelemetryNeed`'s source — now build one: target `"leafB"`, one sensor
path of your choosing, collector `"10.0.0.99:57000"`, a 30-second interval, and reuse
`fx.WINDOW`. Success: it constructs (validation is the test!) and the assert passes.

In [ ]:
my_need = fx.TELEMETRY_NEED    # TODO: replace with your own models.TelemetryNeed(...)

print("kind →", my_need.kind, "| target →", my_need.target,
      "| every", my_need.sample_interval_s, "s")
# assert my_need.target == "leafB" and my_need != fx.TELEMETRY_NEED   # un-comment once built

<details><summary>✅ Solution 7.1 — peek only after trying</summary>

```python
my_need = models.TelemetryNeed(
    target="leafB",
    sensor_paths=["/interface[name=ethernet-1/2]/statistics"],
    collector_endpoint="10.0.0.99:57000",
    sample_interval_s=30,
    window=fx.WINDOW,
)
```

Note what you did *not* type: `v` and `kind` filled themselves in — the `Literal`
defaults §9 explains. Constructing it *is* the test: bad values could not have built.
</details>

The params classes — `BandwidthParams` / `TelemetryParams` — are the *decoded* view of
what lives on the chain as an opaque ABI blob (`fx.BANDWIDTH_PARAMS_ABI`; notebook 04
dissects that blob by hand). Same numbers, friendlier shape:

In [ ]:
print("Ada's need   →", fx.BANDWIDTH_NEED.model_dump())
p = models.BandwidthParams(capacity_bps=fx.CAPACITY_50_MBPS, qos_class=fx.QOS_CLASS)
print("decoded view →", p.model_dump())
assert p.capacity_bps == fx.BANDWIDTH_NEED.capacity_bps == 50_000_000

## 8. The front door: what the package publishes

`a2a_interfaces/__init__.py` is a *facade*: it re-exports the shapes (plus two
`Protocol` ports — the entire subject of notebook 03) and pins them in `__all__`, the
package's published index. The omissions teach as much as the inclusions: the scalar
aliases (`Uint64`, `Address`, …) are internal building blocks, and the fixtures live in
their own namespace (`a2a_interfaces.fixtures`) so example *data* never mixes into the
wire-shape surface.

In [ ]:
import a2a_interfaces

print("published names →", len(a2a_interfaces.__all__), "| first few:", sorted(a2a_interfaces.__all__)[:6])
from a2a_interfaces import Offer
assert Offer is models.Offer                     # a re-export IS the same object

try:
    from a2a_interfaces import Uint64            # building block: deliberately unpublished
except ImportError:
    print("from a2a_interfaces import Uint64 → ✗ ImportError  (reach it via a2a_interfaces.models)")

## 9. `ServiceNeed`: one name, two shapes (the discriminated union)

Bell's server receives a quote request as a loose dict off the wire. Is it a
`BandwidthNeed` or a `TelemetryNeed`? A **discriminated union** answers: a `Union` of
models (01) plus a *discriminator* — the field whose value names the variant. Look at
the real declaration: `ServiceNeed` is a **type alias**, not a class. You cannot call it.

In [ ]:
print(next(line for line in src.splitlines() if line.startswith("ServiceNeed")))
print()
try:
    models.ServiceNeed(kind="bandwidth")
except TypeError as e:
    print("ServiceNeed(...) → ✗ TypeError:", e)

To *parse* with a bare alias you use `TypeAdapter` (from §4). Feed it both canonical
dicts and watch it pick the right class from the `"kind"` value alone:

In [ ]:
need_adapter = TypeAdapter(models.ServiceNeed)

bw = need_adapter.validate_python(fx.BANDWIDTH_NEED.model_dump())
tm = need_adapter.validate_python(fx.TELEMETRY_NEED.model_dump())
print('kind="bandwidth" → parsed as', type(bw).__name__)
print('kind="telemetry" → parsed as', type(tm).__name__)
assert type(bw) is models.BandwidthNeed and type(tm) is models.TelemetryNeed

**✏️ Your turn 9.1 — swap the tag, watch the routing**

Take the telemetry dump but relabel it `kind="bandwidth"` — a telemetry body behind a
bandwidth label. Run it, count the errors, then flip the label back and re-run to see
it parse clean. Success: 8 errors, all `missing` or `extra_forbidden`.

In [ ]:
relabeled = {**fx.TELEMETRY_NEED.model_dump(), "kind": "bandwidth"}   # TODO: after one run, flip back to "telemetry"

caught = None
try:
    routed = need_adapter.validate_python(relabeled)
    print("parsed clean as", type(routed).__name__)
except ValidationError as e:
    caught = e
    print("routed by the tag, then rejected —", e.error_count(), "errors:")
    for err in e.errors():
        print(f"  {'.'.join(str(p) for p in err['loc']):<38} → {err['type']}")
# assert caught.error_count() == 8   # un-comment for the relabeled run

<details><summary>✅ Solution 9.1 — peek only after trying</summary>

```python
# with kind="bandwidth": 4 missing (src, dst, capacity_bps, qos_class)
#                      + 4 extra_forbidden (target, sensor_paths, …) = 8 errors
assert caught.error_count() == 8
```

The discriminator only picks the *door*: the tag routed the dict to `BandwidthNeed`,
and that variant's own rules did all eight rejections.
</details>

An unknown tag is rejected before any other field is even considered. And notice the
trick that makes tagging free: `kind: Literal["bandwidth"] = "bandwidth"` is **pin and
tag at once** — the only legal value is the default, so writers never type it, yet every
dump carries it and the union routes on it.

In [ ]:
try:
    need_adapter.validate_python({**fx.BANDWIDTH_NEED.model_dump(), "kind": "carrier-pigeon"})
except ValidationError as e:
    err = e.errors()[0]
    print("✗ unknown kind →", err["type"])
    print("  msg →", err["msg"][:90] + "…")
    assert err["type"] == "union_tag_invalid"

The same `Literal`-with-default trick pins the **protocol version**: every payload
carries `v: Literal[0] = 0` (module constant `V`). The law behind it is CLAUDE.md rule 3:
if a shape ever changes, bump its `v` **and** update `docs/03-interfaces.md` in the
*same commit* — so a stray `v=1` today is a validation error, never a mystery.

In [ ]:
print("protocol version V →", models.V)
print("carried everywhere → need.v =", fx.BANDWIDTH_NEED.v, "| signed_offer.v =", fx.CANONICAL_SIGNED_OFFER.v)

try:
    models.BandwidthNeed(**{**fx.BANDWIDTH_NEED.model_dump(), "v": 1})
except ValidationError as e:
    print("v=1 → ✗", e.errors()[0]["type"], "  ← a version bump is a deliberate act, never a typo")
    assert e.errors()[0]["type"] == "literal_error"

## 10. The `Offer`: twelve fields, two spellings

The centerpiece. When Bell quotes Ada, it signs an `Offer` — and the *same twelve
fields, in the same order*, are what the smart contract verifies on-chain (the EIP-712
signing format; notebook 07 owns it). Read the source: `Offer` skips `_Frozen` and
declares its own four-knob config — the same `frozen` + `extra="forbid"` guarantees,
**plus** two alias knobs it alone needs.

In [ ]:
print(inspect.getsource(models.Offer))

`alias_generator=to_camel` gives every field a second, camelCase spelling
(`service_type` → `serviceType`); `populate_by_name=True` keeps the snake_case names
usable in Python. Why bother? Because the wire form must mirror the **Solidity struct
byte-for-byte** — what Bell signs must be *exactly* what the contract verifies, down to
the field names (docs/03 §0 calls this the one deliberate camelCase exception; notebook
07 cashes this in). Compare the two dumps:

In [ ]:
offer = fx.CANONICAL_OFFER
snake = offer.model_dump()                    # Python-side spelling
camel = offer.model_dump(by_alias=True)       # wire / Solidity spelling
print(f"  {'model_dump()':<22}   model_dump(by_alias=True)")
for s_key, c_key in zip(snake, camel):
    print(f"  {s_key:<22} → {c_key}" + ("" if s_key == c_key else "   ← renamed"))
assert len(snake) == 12 and "service_type" in snake and "serviceType" in camel

`populate_by_name=True` means *construction* accepts either spelling — hand the
camelCase wire dict straight back to `Offer(**…)` and you get an equal object. Prove,
too, that the `_Frozen` guarantees didn't get lost when `Offer` wrote its own config:

In [ ]:
again = models.Offer(**camel)                    # camelCase in…
print("built from the camel dict == canonical →", again == offer, "✓")
assert again == offer
assert models.Offer.model_config["frozen"] is True
assert models.Offer.model_config["extra"] == "forbid"
print("frozen + forbid still on — Offer only *adds* the two alias knobs")

**✏️ Your turn 10.1 — build the Offer from snake_case**

The demo rebuilt the offer from the *camel* dict. Predict before running: does
`models.Offer(**snake)` — the snake_case dump — also build, and which config knob
decides? Success: the printed outcome matches your prediction and the assert passes.

In [ ]:
# TODO: predict BEFORE running — build, or ValidationError? which knob decides? prediction: ___
try:
    rebuilt = models.Offer(**snake)
    print("Offer(**snake) → built ✓ | equal to canonical →", rebuilt == offer)
except ValidationError as e:
    print("Offer(**snake) → ✗", e.errors()[0]["type"])
# assert models.Offer(**snake) == offer   # un-comment after predicting

<details><summary>✅ Solution 10.1 — peek only after trying</summary>

```python
# prediction: builds — populate_by_name=True accepts the field names
# alongside the camelCase aliases; without that knob, snake keys would
# be rejected because the alias generator owns the official spelling.
assert models.Offer(**snake) == offer
```

`populate_by_name=True` is exactly this permission: aliases for the wire, field names
for Python — both spellings build the same validated object.
</details>

## 11. JSON round-trip: the shape survives the wire

Time to gloss the words properly. **JSON** is the universal text format for structured
data — `{"key": value}` — and it's what all these payloads look like between processes.
To **serialize** is to turn an object into text; to **deserialize** is to parse text
back into an object. Pydantic does each in one call — and because deserializing
*re-validates*, a shape that crosses the wire arrives *proven*, not assumed.

In [ ]:
wire = offer.model_dump_json(by_alias=True)          # serialize: model → JSON text
print("on the wire →", wire[:76] + "…")
back = models.Offer.model_validate_json(wire)        # deserialize + re-validate
print("round-trip intact →", back == offer, "✓")
assert back == offer

**✏️ Your turn 11.1 — tamper with the wire**

Simulate corruption in transit: the tampering below swaps the price for `"10.5"` inside
the JSON text itself. Predict which field's rule fires and the error `type`, then run.
Success: the border refuses and the error name matches §5's decimal-point ban.

In [ ]:
tampered = wire.replace('"' + fx.PRICE_10_TOK + '"', '"10.5"')   # a decimal point sneaks in
# TODO: predict BEFORE running — which loc, which error `type`? prediction: ___
caught = None
try:
    models.Offer.model_validate_json(tampered)
    print("tampered wire accepted?!")
except ValidationError as e:
    caught = e
    print("✗ refused at", e.errors()[0]["loc"], "→", e.errors()[0]["type"])
# assert caught.errors()[0]["type"] == "string_pattern_mismatch"   # un-comment once it fires

<details><summary>✅ Solution 11.1 — peek only after trying</summary>

```python
# ('price',) → string_pattern_mismatch — §5's ^[0-9]+$ bans the dot
assert caught.errors()[0]["type"] == "string_pattern_mismatch"
```

Deserializing *re-validates* — that is the whole point of §11: a payload corrupted
between processes cannot arrive looking proven.
</details>

One shape needs a special trick: `EntitlementView`, the controller's read-only view of a
minted ticket. Its `resource_id` / `terms_hash` are raw `bytes` — Python's type for raw
8-bit values; hex is just how we *print* them. JSON has no bytes, so the config says
`ser_json_bytes="hex"` / `val_json_bytes="hex"`: serialize bytes as hex text, parse hex
text back into bytes. (`terms_hash` is a **hash** — a short, fixed-size fingerprint of a
document; notebook 04 treats hashes properly.)

In [ ]:
print(inspect.getsource(models.EntitlementView))

Poke it on canonical ticket #7. A gotcha worth memorizing: the JSON form is **bare hex,
no `0x` prefix** — comparing against the repo's `0x…` constants needs a `[2:]` slice.

In [ ]:
ev = fx.CANONICAL_ENTITLEMENT_VIEW
print("in Python →", ev.resource_id, f"  ({len(ev.resource_id)} bytes)")
js = ev.model_dump_json()
print("in JSON   →", repr(json.loads(js)["resource_id"][-8:]), " ← last 8 chars: bare hex, no 0x")
assert json.loads(js)["resource_id"] == fx.RESOURCE_ID[2:]     # the [2:] drops '0x'
assert models.EntitlementView.model_validate_json(js) == ev
print("bytes → hex → bytes round-trip ✓")

## 12. The mutable default trap (and how pydantic dodges it)

`DashboardEvent` — the JSONL event lines behind the live run view (notebook 11) — has a
field default that should make a Python veteran flinch: `detail: dict[str, Any] = {}`.
In plain Python, a mutable default is a famous bug: the default is created **once** and
then *shared by every instance*. Watch the trap spring in a plain class first:

In [ ]:
class PlainEvent:
    def __init__(self, detail={}):      # ← created ONCE at def-time, shared forever
        self.detail = detail


a, b = PlainEvent(), PlainEvent()
a.detail["leak"] = True                  # touch only a…
print("b.detail →", b.detail, "  ← …and b is contaminated")
assert a.detail is b.detail              # literally the same dict object

Pydantic **deep-copies the default for every instance**, so the identical-looking
declaration is safe. (Honest footnote: we mutate `e1.detail` below even though the model
is frozen — `frozen` stops *rebinding fields*, not mutating the innards of a container
you already hold. The guarantee covers the model's fields, not a dict's contents.)

In [ ]:
def mk():
    return models.DashboardEvent(ts=fx.WINDOW.start, step="fulfill",
                                 trust_domain="chain", narration="ticket #7 minted")


e1, e2 = mk(), mk()
e1.detail["leak"] = True
print("e2.detail →", e2.detail, "  ← clean: each instance got its own fresh copy")
assert e1.detail is not e2.detail and e2.detail == {}

**✏️ Your turn 12.1 — fix PlainEvent the classic way**

Pydantic dodged the trap for you — now fix plain Python yourself with the classic
None-sentinel idiom: default the parameter to `None` and create a fresh `{}` inside
`__init__`. Success: the second instance stays clean and the assert passes.

In [ ]:
class SafeEvent:
    def __init__(self, detail={}):      # TODO: default to None, make a fresh {} inside
        self.detail = detail


sa, sb = SafeEvent(), SafeEvent()
sa.detail["leak"] = True
print("sb.detail →", sb.detail, "  ← {} means fixed; {'leak': True} means not yet")
# assert sa.detail is not sb.detail and sb.detail == {}   # un-comment once fixed

<details><summary>✅ Solution 12.1 — peek only after trying</summary>

```python
class SafeEvent:
    def __init__(self, detail=None):
        self.detail = detail if detail is not None else {}   # fresh dict per call
```

`None` is immutable, so sharing it is harmless; the fresh `{}` is created at *call*
time, not def time — the hand-rolled version of what pydantic does for every field.
</details>

## 13. The shape as a publishable document: JSON Schema

A **schema** is a machine-readable description of a shape — which fields exist, their
types, their constraints. Every pydantic model can export one (`json_schema()`), which
means the repo's *entire published language* can be handed to another team, another
programming language, or an LLM as a formal document. Check that §4's chain-width edge
and §9's discriminator both survive into the export:

In [ ]:
schema = need_adapter.json_schema()
print("top level →", list(schema.keys()))
print("variants  →", list(schema["$defs"].keys()))
cap = schema["$defs"]["BandwidthNeed"]["properties"]["capacity_bps"]
print("capacity_bps →", cap)
assert cap["maximum"] == 2**64 - 1                          # the chain's width, published
assert schema["discriminator"]["propertyName"] == "kind"    # the tag, published

## 14. Finale: the hole in the armor — `model_copy` skips validation

Frozen models still need a way to make *variants* (say, a revoked copy of a view for a
test) — that's `model_copy(update={…})`. But here is the sharpest edge in all of
pydantic: **`model_copy` does not re-validate**. Watch a frozen, "validated" model
silently accept garbage:

In [ ]:
bad = fx.BANDWIDTH_NEED.model_copy(update={"capacity_bps": "oops"})
print("no exception raised. capacity_bps is now →", repr(bad.capacity_bps))
assert bad.capacity_bps == "oops"        # a 'validated', frozen model — holding a string

Why is this allowed? Validation is a *boundary* activity, and a copy never crossed one —
pydantic assumes in-process code knows what it's doing. The repo's discipline follows
directly: `model_copy` appears only for **local** variants headed straight into a pure
function (e.g. `skeleton_explore.ipynb` flips `revoked=True` to test the predicate) —
**never** for data about to cross a package boundary. Boundaries re-validate, and
re-validation catches the rot. (Pydantic may even grumble a serializer warning while
dumping the corrupt model — the dump itself smells it.)

In [ ]:
try:
    models.BandwidthNeed.model_validate(bad.model_dump())
except ValidationError as e:
    print("✗ re-validation at the border →", e.errors()[0]["type"])
    assert e.errors()[0]["type"] == "int_parsing"
print("moral: guarantees hold at CONSTRUCTION — anything that crossed a boundary gets re-validated")

**✏️ Your turn 14.1 — make the disciplined copy**

Do the *legitimate* `model_copy`: flip ticket #7's view to `revoked=True` — the exact
move `skeleton_explore` uses to test the predicate. Observe: the copy changes, the
canonical fixture does not, and the well-formed copy sails through border re-validation.

In [ ]:
revoked_view = fx.CANONICAL_ENTITLEMENT_VIEW    # TODO: .model_copy(update={"revoked": True})

print("copy revoked?   →", revoked_view.revoked)
print("fixture intact? →", fx.CANONICAL_ENTITLEMENT_VIEW.revoked is False)
models.EntitlementView.model_validate(revoked_view.model_dump())   # the border test
print("border re-validation → ✓  (well-formed variants survive; §14's garbage would not)")
# assert revoked_view.revoked and not fx.CANONICAL_ENTITLEMENT_VIEW.revoked   # un-comment once copied

<details><summary>✅ Solution 14.1 — peek only after trying</summary>

```python
revoked_view = fx.CANONICAL_ENTITLEMENT_VIEW.model_copy(update={"revoked": True})
assert revoked_view.revoked and not fx.CANONICAL_ENTITLEMENT_VIEW.revoked
```

This is the discipline in one cell: `model_copy` for a *local* variant headed into a
pure function, and re-validation waiting at any boundary it might later cross.
</details>

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| Watched a typo'd dict lie silently (§1), then explode loudly (§6) | validation lives at the boundary | every package border in this repo |
| Built `Ticket`, read `loc`/`type`/`msg`/`input` | `BaseModel` + `ValidationError` anatomy | 08's FastAPI turns HTTP bodies into these models automatically |
| Hit the `2**64 − 1` edge | `Uint*` aliases mirror the chain's integer widths | 06 (Solidity's `uint64`), 04 (the numbers by hand) |
| Counted 40 and 130 hex chars; banned the decimal point | `Address`/`Signature`/`DecimalString` patterns | 06 (addresses), 07 (real 65-byte signatures) |
| Had mutation and typos refused | `_Frozen`: `frozen=True` + `extra="forbid"`, inherited once | shared fixtures no test can poison |
| Let `"kind"` pick the class | discriminated union + `TypeAdapter` | 10's agents parse quotes off the A2A wire |
| Dumped the `Offer` in two spellings | `alias_generator=to_camel` + `populate_by_name` | 07 signs the camelCase mirror of the Solidity struct |
| Round-tripped JSON; bytes as bare hex | `model_dump_json`/`model_validate_json`, `ser_json_bytes="hex"` | 07/08 pass `EntitlementView`s around |
| Sprang the mutable-default trap — then didn't | pydantic deep-copies field defaults per instance | 11's `DashboardEvent` stream |
| Corrupted a frozen model with `model_copy` | guarantees hold at construction only | why boundaries re-validate, everywhere |

## Experiments to try (predict first, then run)

- Build `models.TimeWindow(start=10, end=5)` — an *inverted* window. Predict: error or
  not? Then sit with what the answer means: pydantic checks *shape*, not *sense* — whose
  job is `start < end`? (You'll meet the answer in notebook 08's predicate.)
- Dump the canonical offer with plain `model_dump_json()` (snake_case) and feed that to
  `Offer.model_validate_json`. Predict: does the round-trip still work — and which
  config knob decides?
- Give a `DashboardEvent` `trust_domain="bank"`. Predict the error `type` before you run.
- Deliberate breakage: in §9, `pop("kind")` out of the dict before `validate_python`.
  Predict the error type — is it the same as carrier-pigeon's? (Hint: *not-found* ≠
  *invalid*.)

## Check yourself

1. A payload arrives with one extra, unknown field. What happens at construction, and
   which config knob decides?
2. Why does `price` travel as `"10000000000000000000"` — a string — rather than a float
   `10.0` or a plain number?
3. `TypeAdapter(models.ServiceNeed).validate_python(d)` returns a `TelemetryNeed`. What,
   exactly, in `d` made that decision?
4. Which dump goes on the wire — `model_dump()` or `model_dump(by_alias=True)` — and why
   must the wire spelling match Solidity's?
5. A colleague "fixes" a model with `model_copy(update=…)` and ships it to another
   package. Which guarantee did they silently drop, and what must the receiving side do?

**Where this goes next:** `03_protocols_and_ports.ipynb` — the *other* half of the
published language: the two `typing.Protocol` ports (`EntitlementReader`,
`NetworkProvisioner`) you glimpsed in `__all__` back in §8.

**Deeper dive**

- [`docs/03-interfaces.md`](../../../docs/03-interfaces.md) — the published language, schema by schema
- [`e2e/notebooks/skeleton_explore.ipynb`](../skeleton_explore.ipynb) — these shapes driving the whole cardboard lifecycle
- [`interfaces/src/a2a_interfaces/models.py`](../../../interfaces/src/a2a_interfaces/models.py) — the 323 lines you have now fully read